# ADS 层 — KPI 指标

从 DWS 聚合产出核心业务指标：
- **trip_count**：各来源各日行程总量
- **GMV**：总流水金额
- **govt_surcharge_rate**：政府附加费率（MTA 税 + 改善费 + 拥堵费 + 机场费 / 车费）；
  注意这是政府税费比例，非平台抽成；FHV 和出租车的附加费结构不同，不宜直接对比
- **fhv_platform_take_rate**：FHV HV 平台抽成率 `1 - driver_pay / fare`（仅 FHV HV 有效）
- **tip_rate**：小费 / 车费
- **avg_fare、avg_distance、avg_duration_min**：效率指标


## 1. SparkSession

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TLC-ADS-Metrics")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")  # only overwrite TARGET_DATE partition
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE    = os.path.expanduser("~/tlc-spark-pipeline")
DWS_IN  = f"{BASE}/data/warehouse/dws/trips_daily"
ADS_OUT = f"{BASE}/data/warehouse/ads/kpi_daily"
TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready | TARGET_DATE={TARGET_DATE}")


## 2. 日度 KPI 汇总

In [ ]:
dws = spark.read.parquet(DWS_IN).filter(
    (F.col("pickup_year") == yr) & (F.col("pickup_month") == mo)
    & (F.col("pickup_day") == day))

daily_kpi = (
    dws
    .groupBy("pickup_year", "pickup_month", "pickup_day", "source_type")
    .agg(
        F.sum("trip_count").alias("trip_count"),
        F.round(F.sum("gmv"), 2).alias("gmv"),
        F.round(F.sum("fare_sum"), 2).alias("fare_sum"),
        F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
        F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
        F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
        F.round(F.sum("duration_sum"), 2).alias("duration_sum"),
        F.round(F.sum("driver_pay_sum"), 2).alias("driver_pay_sum")
    )
    # 政府附加费率 = surcharge / fare（加权比例：sum/sum，不是 avg(rate)）
    # surcharge 是 MTA税+改善费+拥堵费+机场费，是政府税费，不是平台抽成
    .withColumn("govt_surcharge_rate",
        F.round(F.col("surcharge_sum") / F.col("fare_sum"), 4))
    # 小费率
    .withColumn("tip_rate",
        F.round(F.col("tip_sum") / F.col("fare_sum"), 4))
    # FHV HV 平台抽成率：1 - 司机实得 / 乘客基础票价
    # driver_pay_sum 对出租车为 null（出租车无此字段），FHV HV 有效
    .withColumn("fhv_platform_take_rate",
        F.when(
            F.col("driver_pay_sum").isNotNull() & (F.col("fare_sum") > 0),
            F.round(1 - F.col("driver_pay_sum") / F.col("fare_sum"), 4)
        ).otherwise(F.lit(None)))
    # 均值指标
    .withColumn("avg_fare",
        F.round(F.col("fare_sum") / F.col("trip_count"), 2))
    .withColumn("avg_distance_mi",
        F.round(F.col("distance_sum") / F.col("trip_count"), 2))
    .withColumn("avg_duration_min",
        F.round(F.col("duration_sum") / F.col("trip_count"), 1))
    .orderBy("pickup_year", "pickup_month", "pickup_day", "source_type")
)

print("=== 日度 KPI ===")
daily_kpi.select(
    "pickup_day", "source_type", "trip_count", "gmv",
    "govt_surcharge_rate", "fhv_platform_take_rate", "tip_rate", "avg_fare"
).show(20)


## 3. 写出 ADS

In [ ]:
%%time
# 动态分区覆盖：每次只覆盖 TARGET_DATE 对应分区，历史数据不受影响
(
    daily_kpi
    .write
    .partitionBy("pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(ADS_OUT)
)
print(f"ADS written to: {ADS_OUT}")

# 汇总（全量 — 使用加权 sum/sum 而非 avg(rate)）
print("\n=== 全量汇总 ===")
total = daily_kpi.groupBy("source_type").agg(
    F.sum("trip_count").alias("total_trips"),
    F.round(F.sum("gmv"), 0).alias("total_gmv"),
    # 政府附加费率：加权聚合
    F.round(F.sum("surcharge_sum") / F.sum("fare_sum") * 100, 1).alias("govt_surcharge_rate_pct"),
    F.round(F.sum("tip_sum") / F.sum("fare_sum") * 100, 1).alias("tip_rate_pct"),
    # FHV 平台抽成率：加权聚合（出租车为 null）
    F.round((1 - F.sum("driver_pay_sum") / F.sum("fare_sum")) * 100, 1).alias("platform_take_rate_pct")
)
total.show()

spark.stop()
print("ADS metrics complete.")
